# Tahap 1: Pembersihan dan Standardisasi Dataset Resep
Notebook ini membersihkan data resep, menangani duplikasi dan missing values, menstandarkan kolom porsi dan nutrisi, serta menghasilkan ID unik recipe_id.

In [1]:
import os
import re
import glob
import sys
import pandas as pd

if sys.stdout.encoding != 'utf-8':
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

In [2]:
def clean_text(val):
    if pd.isna(val):
        return ""
    text = str(val).strip()
    return re.sub(r'\s+', ' ', text)

def parse_portions(val):
    if pd.isna(val):
        return 1
    val_str = str(val).strip().lower()
    match = re.search(r'(\d+)', val_str)
    if match:
        porsi = int(match.group(1))
        return porsi if porsi > 0 else 1
    return 1

def to_float(val):
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip().replace(',', '.')
    match = re.search(r'[-+]?\d*\.?\d+', val_str)
    if match:
        try:
            return round(float(match.group(0)), 2)
        except ValueError:
            return 0.0
    return 0.0

In [3]:
csv_files = glob.glob(os.path.join("dataset", "*.csv"))
csv_files = [f for f in csv_files if "cleaned_recipes.csv" not in os.path.basename(f)]
print(f"File CSV yang ditemukan: {[os.path.basename(f) for f in csv_files]}")

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    print(f"Membaca {os.path.basename(f)}: {df.shape[0]} baris, {df.shape[1]} kolom")
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
print(f"Total baris awal: {len(combined)}")

File CSV yang ditemukan: ['diet_mate_recipes.csv', 'nourish_recipes.csv']
Membaca diet_mate_recipes.csv: 319 baris, 10 kolom
Membaca nourish_recipes.csv: 21 baris, 10 kolom
Total baris awal: 340


In [4]:
combined.columns = [c.strip().lower() for c in combined.columns]
rename_map = {
    'kategori': 'kategori',
    'nama_masakan': 'nama_masakan',
    'hasil': 'jumlah_porsi_raw',
    'kalori_kkal': 'kalori_kkal',
    'protein_g': 'protein_g',
    'karbohidrat_g': 'karbohidrat_g',
    'lemak_g': 'lemak_g',
    'bahan_bahan': 'bahan_bahan',
    'cara_memasak': 'cara_memasak',
    'url': 'url'
}
combined = combined.rename(columns=rename_map)

for col in ['nama_masakan', 'kategori', 'bahan_bahan', 'cara_memasak', 'url']:
    if col in combined.columns:
        combined[col] = combined[col].apply(clean_text)

combined = combined[combined['nama_masakan'] != ""]
combined['name_lower'] = combined['nama_masakan'].str.lower()
dup_count = combined.duplicated(subset=['name_lower']).sum()
combined = combined.drop_duplicates(subset=['name_lower'], keep='first').drop(columns=['name_lower'])
print(f"Jumlah duplikat dihapus: {dup_count}")

combined['jumlah_porsi'] = combined['jumlah_porsi_raw'].apply(parse_portions).astype(int)
combined['basis_porsi'] = "1 porsi"

for col in ['kalori_kkal', 'protein_g', 'karbohidrat_g', 'lemak_g']:
    combined[col] = combined[col].apply(to_float).astype(float)

combined = combined.reset_index(drop=True)
combined['recipe_id'] = [f"RCP_{i+1:04d}" for i in range(len(combined))]

final_cols = [
    'recipe_id', 'nama_masakan', 'kategori', 'jumlah_porsi', 'basis_porsi',
    'kalori_kkal', 'protein_g', 'karbohidrat_g', 'lemak_g',
    'bahan_bahan', 'cara_memasak', 'url'
]
cleaned_df = combined[final_cols]
print(f"Total resep valid setelah pembersihan: {len(cleaned_df)}")

Jumlah duplikat dihapus: 13
Total resep valid setelah pembersihan: 327


In [5]:
output_path = os.path.join("dataset", "cleaned_recipes.csv")
cleaned_df.to_csv(output_path, index=False, encoding='utf-8')
print(f"Data tersimpan di {output_path}")
cleaned_df[['recipe_id', 'nama_masakan', 'jumlah_porsi', 'basis_porsi', 'kalori_kkal', 'protein_g']].head(5)

Data tersimpan di dataset\cleaned_recipes.csv


,recipe_id,nama_masakan,jumlah_porsi,basis_porsi,kalori_kkal,protein_g
0,RCP_0001,Toge Telur,1,1 porsi,252.0,21.90
1,RCP_0002,Tumis Labu Siam,5,1 porsi,61.0,0.79
2,RCP_0003,Dimsum Ayam Udang,27,1 porsi,50.0,3.70
3,RCP_0004,Bakso Ayam Oat,4,1 porsi,197.0,19.25
4,RCP_0005,Kimchi Jiggae,1,1 porsi,210.0,26.12
